# Execution Pipeline — Orchestrator

Runs the full ingestion-to-results chain in dependency order, skipping any stage
whose outputs are already fresh. This is the entry point: every result in the
thesis is reproducible by running this notebook from a clean checkout.

## Objective
Execute fifteen notebooks across five stages, in the only order their data
dependencies allow, and re-run the minimum needed after a change.

## Pipeline Position

| Stage | Notebooks | Produces |
|---|---|---|
| 0 — Transformation | `01_energy_prices`, `02_weather` | cleaned GME price and ERA5/CAMS weather series |
| 1 — Modelling | `01_solar_irradiation`, `02_temperature`, `03_noct_uplift`, `04_solar_share_historical`, `05_capture_rate`, `06_copula` | fitted marginal models, the thermal uplift profile, and the dependence structure |
| 2 — Scenario Paths | `01_temperature_paths`, `02_solar_share_paths` | deterministic SSP temperature and solar-penetration trajectories |
| 3 — Simulation | `01_innovations`, `02_variables_reconstruction` | Monte Carlo innovations, then physical variable paths per penetration scenario |
| 4 — Results | `03_revenue_index_historical`, `04_revenue_index_scenarios`, `05_hedge_contract` | the historical benchmark, the nine-scenario revenue index, and the hedge |

Stage 3 and Stage 4 notebooks share one numbering sequence because they live in
one folder and run as one continuous chain; the stage boundary marks where
simulation ends and results begin.

## Decision Log

| # | Decision | Alternatives | Rationale | Impact |
|---|---|---|---|---|
| D1 | Staleness is decided per stage, not per notebook | Track each notebook's own inputs and outputs | Notebooks within a stage share intermediate state and are only meaningful as a group; a per-notebook graph would need every intermediate declared as an artifact | A change anywhere in a stage re-runs that whole stage. Stages are small enough that this costs little |
| D2 | A stage is stale if any of its **own notebooks** is newer than its outputs, not only if its data inputs are | Compare declared upstream data files alone | Refactoring a computation in place changes no data file, so an input-only rule would leave the outputs stale and silently wrong | Editing comments or markdown also marks a stage stale. That is the safe direction to err in |
| D3 | A stage that re-runs marks every downstream stage stale (cascade) | Re-check each downstream stage against mtimes | mtime comparison alone is fragile across filesystems and clock skew, and a stage that just wrote its outputs would look fresh to its successor | The cascade is unconditional, so a Stage-0 change always rebuilds everything |
| D4 | Each notebook runs in a **fresh kernel process**, driven by `nbclient` in-process | One shared kernel; or shelling out to `jupyter nbconvert` | The simulation notebooks hold multi-gigabyte arrays, so a shared kernel would accumulate them across the chain and exhaust memory. `nbconvert` as a subprocess exits 1 with empty stdout and stderr when spawned from inside a kernel, which is how this orchestrator itself runs | Results are written back into each notebook, success or failure, so the committed notebooks show the run that produced the current artifacts and a failure keeps its traceback |
| D5 | Shared modules (`model_utils.py`, `risk_metrics.py`) are declared as stage inputs | Treat them as code, outside the dependency graph | They carry the production chain, the path-layout guards and the risk metrics, so editing one changes results exactly as editing a notebook does | Touching a shared module re-runs every stage that imports it |

**Shared code.** `Code/model_utils.py` holds the production chain, the thermal
uplift mapper and the simulation path guards. `Code/risk_metrics.py` holds
VaR/CVaR, variance reduction, drawdown and Sortino. Neither is a stage; both are
declared as inputs wherever they are imported.

**To force a full rebuild**, set `FORCE_RERUN = True`. To preview without
executing, set `DRY_RUN = True`.

In [1]:
import sys, time
from pathlib import Path

sys.path.append(str(Path(__file__).parent if '__file__' in dir()
                    else Path.cwd()))
from model_utils import find_project_root
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

## Configuration
Edit the flags below before running the pipeline.

In [2]:
# ── Run behaviour ──────────────────────────────────────────────────────────────
FORCE_RERUN      = False   # True: run all stages regardless of staleness
DRY_RUN          = False   # True: print plan only, do not execute notebooks
NOTEBOOK_TIMEOUT = 7200    # max seconds per notebook (2 h); increase for large sims

# Run a subset of stages (0-based indices), or None for all.
#   0 Transformation   1 Modelling   2 Scenario Paths   3 Simulation   4 Results
# Example: STAGE_FILTER = [3, 4]  -> run only Simulation and Results.
# NOTE: filtering out an upstream stage also suppresses its cascade, so a subset
# run trusts that whatever it skipped is already fresh.
STAGE_FILTER = None

In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# Anchored on the directory that actually holds Code/ and Data/, not on the
# cwd: a bare Path('..') made every stage path depend on where the orchestrator
# was launched from, and silently reported every output as missing when it was
# launched from anywhere else.
ROOT = find_project_root()
CODE = ROOT / 'Code'
DATA = ROOT / 'Data'

print(f'Project root : {ROOT}')
print(f'Python       : {sys.executable}')
print()

# ── Stage definitions ──────────────────────────────────────────────────────────
# notebooks : paths relative to Code/; executed in the listed order
# inputs    : files whose mtime is compared against output mtimes
# outputs   : files that must exist and be up-to-date; missing → stale

STAGES = [
    {
        'name': '0 - Transformation',
        'notebooks': [
            'Transformation/01_energy_prices.ipynb',
            'Transformation/02_weather.ipynb',
        ],
        'inputs': [
            DATA / 'Raw' / 'MGP_Prezzi2005010120260322.zip',
            DATA / 'Raw' / 'reanalysis-era5-single-levels-timeseries-sfc7rupio5o.nc',
            DATA / 'Raw' / 'cams_solar_irradiance_ts.nc',
        ],
        'outputs': [
            DATA / 'Cleaned' / 'df_energy_cleaned.parquet',
            DATA / 'Cleaned' / 'df_bologna_cleaned.parquet',
        ],
    },
    {
        'name': '1 - Modelling',
        'notebooks': [
            'Modelling/01_solar_irradiation.ipynb',
            'Modelling/02_temperature.ipynb',
            # 03_noct_uplift is the ONLY producer of noct_uplift_params.pkl,
            # which four Stage-3/4 notebooks read. It has no dependency on the
            # other Stage-1 fits, only on the cleaned ERA5 record.
            'Modelling/03_noct_uplift.ipynb',
            'Modelling/04_solar_share_historical.ipynb',
            # 05_capture_rate MUST precede 06_copula: the latter consumes
            # df_cr_modelled.parquet and cr_ar_params.pkl produced here.
            'Modelling/05_capture_rate.ipynb',
            'Modelling/06_copula.ipynb',
        ],
        'inputs': [
            DATA / 'Cleaned' / 'df_bologna_cleaned.parquet',
            DATA / 'Cleaned' / 'df_energy_cleaned.parquet',
            CODE / 'model_utils.py',    # D5: shared helpers and constants
        ],
        'outputs': [
            DATA / 'Modelled'  / 'df_solar_modelled.parquet',
            DATA / 'Modelled'  / 'df_temperature_modeled.parquet',
            DATA / 'Modelled'  / 'df_cr_modelled.parquet',
            DATA / 'Cleaned'   / 'df_solar_share.parquet',
            CODE / 'Models'    / 'GHI - KT'    / 'ghi_arma_result.pkl',
            CODE / 'Models'    / 'Temperature'  / 'temp_model_params.pkl',
            CODE / 'Models'    / 'Copula'       / 'copula_params.pkl',
            CODE / 'Models'    / 'Capture rate' / 'cr_ar_params.pkl',
            CODE / 'Models'    / 'Temperature'  / 'noct_uplift_params.pkl',
        ],
    },
    {
        'name': '2 - Scenario Paths',
        'notebooks': [
            'Scenarios/01_temperature_paths.ipynb',
            'Scenarios/02_solar_share_paths.ipynb',
        ],
        'inputs': [
            CODE / 'Models' / 'Temperature' / 'temp_model_params.pkl',
            DATA / 'Cleaned' / 'df_solar_share.parquet',
        ],
        'outputs': [
            DATA / 'Cleaned' / 'df_temperature_scenarios.parquet',
            DATA / 'Cleaned' / 'df_penetration_scenarios.parquet',
        ],
    },
    {
        'name': '3 - Simulation',
        'notebooks': [
            'Simulation/01_innovations.ipynb',
            'Simulation/02_variables_reconstruction.ipynb',
        ],
        'inputs': [
            CODE / 'Models' / 'GHI - KT'    / 'ghi_arma_result.pkl',
            CODE / 'Models' / 'Temperature'  / 'temp_model_params.pkl',
            CODE / 'Models' / 'Copula'       / 'copula_params.pkl',
            CODE / 'Models' / 'Capture rate' / 'cr_ar_params.pkl',
            DATA / 'Modelled' / 'df_solar_modelled.parquet',
            DATA / 'Modelled' / 'df_temperature_modeled.parquet',
            DATA / 'Modelled' / 'df_cr_modelled.parquet',
            DATA / 'Cleaned'  / 'df_penetration_scenarios.parquet',
            CODE / 'model_utils.py',    # D5
        ],
        'outputs': [
            DATA / 'Simulated' / 'mc_innovations.parquet',
            DATA / 'Simulated' / 'mc_innovations_metadata.pkl',
            DATA / 'Simulated' / 'mc_physical_variables_current_trend.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_pniec.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_entso_e.parquet',
        ],
    },
    {
        'name': '4 - Results',
        'notebooks': [
            'Simulation/03_revenue_index_historical.ipynb',
            'Simulation/04_revenue_index_scenarios.ipynb',
            'Simulation/05_hedge_contract.ipynb',
        ],
        'inputs': [
            DATA / 'Simulated' / 'mc_physical_variables_current_trend.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_pniec.parquet',
            DATA / 'Simulated' / 'mc_physical_variables_entso_e.parquet',
            DATA / 'Cleaned'   / 'df_temperature_scenarios.parquet',
            DATA / 'Modelled'  / 'df_cr_modelled.parquet',
            DATA / 'Modelled'  / 'df_temperature_modeled.parquet',
            DATA / 'Cleaned'   / 'df_bologna_cleaned.parquet',
            CODE / 'Models' / 'Temperature' / 'noct_uplift_params.pkl',
            CODE / 'risk_metrics.py',   # shared by all 3 notebooks in this stage
            CODE / 'model_utils.py',    # production chain and path guards
        ],
        'outputs': [
            DATA / 'Results' / 'ri_annual_results_hist.parquet',
            DATA / 'Results' / 'ri_annual_results_multiscenario.parquet',
            DATA / 'Results' / 'risk_metrics_summary.parquet',
        ],
    },
]

print(f'Stages defined: {len(STAGES)}')
total_nbs = sum(len(s['notebooks']) for s in STAGES)
print(f'Notebooks total: {total_nbs}')

Project root : C:\Users\LucasMonero\OneDrive - EloGroup\Documentos\data projects\Master Thesis\Project
Python       : C:\Users\LucasMonero\OneDrive - EloGroup\Documentos\data projects\Master Thesis\Project\.venv\Scripts\python.exe

Stages defined: 5
Notebooks total: 15


## Helper functions

In [4]:
def _mtime(path):
    p = Path(path)
    return p.stat().st_mtime if p.exists() else 0.0


# D1: staleness is decided per STAGE, not per notebook -- see is_stale below,
# which takes one stage at a time.
def is_stale(stage, upstream_reran=False):
    """Return (stale: bool, reason: str)."""
    # D3: a stage that reran upstream is unconditionally stale (the cascade).
    if upstream_reran:
        return True, 'upstream stage reran'

    outputs   = [Path(p) for p in stage.get('outputs', [])]
    inputs    = [Path(p) for p in stage.get('inputs',  [])]
    notebooks = [CODE / nb_rel for nb_rel in stage.get('notebooks', [])]

    missing = [p.name for p in outputs if not p.exists()]
    if missing:
        return True, 'missing output(s): ' + ', '.join(missing)

    # D2: a stage is also stale if one of its OWN notebooks was edited more
    # recently than its outputs — comparing only declared upstream data
    # files misses the case where notebook code changed but its data
    # inputs did not (e.g. refactoring a computation in place).
    if outputs:
        all_inputs  = inputs + notebooks
        existing_in = [p for p in all_inputs if p.exists()]
        if existing_in:
            oldest_out = min(_mtime(p) for p in outputs)
            newest_in  = max(_mtime(p) for p in existing_in)
            if newest_in > oldest_out:
                newer = [p.name for p in existing_in if _mtime(p) > oldest_out]
                return True, 'input(s)/notebook(s) updated since last run: ' + ', '.join(newer)

    return False, 'up to date'


# D4: each notebook runs in its own fresh kernel process (see below).
def run_notebook(nb_rel, timeout=NOTEBOOK_TIMEOUT):
    """Execute one notebook in place, in its own kernel.

    D4: nbclient starts a fresh kernel PROCESS per notebook and shuts it down
    afterwards, so the multi-gigabyte arrays the simulation notebooks hold are
    returned to the OS between stages rather than accumulating.

    Driven in-process rather than through `jupyter nbconvert`: that command
    exits 1 with empty stdout AND stderr when spawned from inside a Jupyter
    kernel, which is how this orchestrator is itself run, so a failure there
    reported nothing usable. nbclient raises the real CellExecutionError.

    Outputs are written back whether the run succeeds or fails, so a failed
    notebook keeps the traceback that explains it.

    Returns (success: bool, elapsed_s: float, error_msg: str).
    """
    nb_path = CODE / nb_rel
    if not nb_path.exists():
        return False, 0.0, 'notebook not found: ' + str(nb_path)

    t0 = time.time()
    nb_obj = nbformat.read(nb_path, as_version=4)
    client = NotebookClient(
        nb_obj,
        timeout=timeout,
        kernel_name='python3',
        allow_errors=False,
        # kernel CWD = the notebook's own directory, matching how a human runs it
        resources={'metadata': {'path': str(nb_path.parent)}},
    )

    ok, err = True, ''
    try:
        client.execute()
    except CellExecutionError as e:
        ok, err = False, str(e)
    except Exception as e:
        ok, err = False, f'{type(e).__name__}: {e}'
    finally:
        nbformat.write(nb_obj, nb_path)

    return ok, time.time() - t0, err[-3000:]


def fmt_time(s):
    if s >= 60:
        return f'{s/60:.1f} min'
    return f'{s:.0f}s'


print('Helper functions ready.')

Helper functions ready.


## Run pipeline

In [5]:
SEP = '=' * 68

print(SEP)
print(f'  PIPELINE  {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  FORCE_RERUN={FORCE_RERUN}  DRY_RUN={DRY_RUN}  '
      f'STAGE_FILTER={STAGE_FILTER if STAGE_FILTER is not None else "all"}')
print(SEP)

log            = []
upstream_reran = False
pipeline_ok    = True

for i, stage in enumerate(STAGES):

    tag = f'[{i+1}/{len(STAGES)}]'

    # ── stage filter ──────────────────────────────────────────────────────────
    if STAGE_FILTER is not None and i not in STAGE_FILTER:
        print(f'\n{tag} {stage["name"]}  -->  FILTERED OUT')
        log.append({'stage': stage['name'], 'status': 'filtered', 'notebooks': []})
        upstream_reran = False
        continue

    # ── staleness check ───────────────────────────────────────────────────────
    stale, reason = is_stale(stage, upstream_reran=upstream_reran)

    if not FORCE_RERUN and not stale:
        print(f'\n{tag} {stage["name"]}  -->  UP TO DATE  ({reason})')
        log.append({'stage': stage['name'], 'status': 'skipped',
                    'reason': reason, 'notebooks': []})
        upstream_reran = False
        continue

    run_reason = reason if not FORCE_RERUN else 'FORCE_RERUN=True'
    print(f'\n{tag} {stage["name"]}  -->  RUNNING  ({run_reason})')

    stage_entry = {
        'stage': stage['name'], 'status': 'ran',
        'reason': run_reason, 'notebooks': [], 'success': True,
    }
    stage_ok = True

    for nb_rel in stage['notebooks']:
        nb_name = Path(nb_rel).name

        if DRY_RUN:
            print(f'  [DRY]  {nb_name}')
            stage_entry['notebooks'].append({'notebook': nb_name, 'status': 'dry_run'})
            continue

        print(f'  Running  {nb_name} ...', end='', flush=True)
        ok, elapsed, err = run_notebook(nb_rel)
        status = 'OK' if ok else 'FAILED'
        print(f'  [{status}]  {fmt_time(elapsed)}')

        nb_entry = {'notebook': nb_name, 'status': status,
                    'elapsed_s': round(elapsed, 1)}
        if not ok:
            nb_entry['error'] = err[-600:]
            print(f'  -- error tail --')
            print(err[-500:])
            print(f'  ----------------')
            stage_ok = False

        stage_entry['notebooks'].append(nb_entry)

        if not ok:
            print(f'  Stopping stage after failure in {nb_name}.')
            break

    stage_entry['success'] = stage_ok
    log.append(stage_entry)
    upstream_reran = stage_ok and not DRY_RUN

    if not stage_ok:
        print(f'\nPipeline halted at stage {stage["name"]}.')
        pipeline_ok = False
        break

print(f'\n{SEP}')
print(f'  {"DONE" if pipeline_ok else "FAILED"}  '
      f'{datetime.now().strftime("%H:%M:%S")}')
print(SEP)

  PIPELINE  2026-09-13 04:33:20
  FORCE_RERUN=False  DRY_RUN=False  STAGE_FILTER=all



[1/5] 0 - Transformation  -->  RUNNING  (input(s)/notebook(s) updated since last run: 01_energy_prices.ipynb, 02_weather.ipynb)
  Running  01_energy_prices.ipynb ...

  [OK]  2.2 min
  Running  02_weather.ipynb ...

  [OK]  15s

[2/5] 1 - Modelling  -->  RUNNING  (upstream stage reran)
  Running  01_solar_irradiation.ipynb ...

  [OK]  4.0 min
  Running  02_temperature.ipynb ...

...

  [OK]  5.2 min
  Running  02_variables_reconstruction.ipynb ...

  [OK]  5.5 min

[5/5] 4 - Results  -->  RUNNING  (upstream stage reran)
  Running  03_revenue_index_historical.ipynb ...

  [OK]  24s
  Running  04_revenue_index_scenarios.ipynb ...

  [OK]  3.7 min
  Running  05_hedge_contract.ipynb ...

  [OK]  2.6 min

  DONE  05:05:41


## Summary

In [6]:
print('PIPELINE SUMMARY')
print('-' * 55)

total_s = sum(
    nb.get('elapsed_s', 0)
    for entry in log
    for nb in entry.get('notebooks', [])
)

STATUS_TAG = {'ran': 'RAN', 'skipped': '---', 'filtered': '   '}

for entry in log:
    s   = entry['status']
    tag = STATUS_TAG.get(s, '???')
    ok  = '' if s != 'ran' else ('  OK' if entry.get('success') else '  FAILED')
    print(f'  [{tag}]  {entry["stage"]}{ok}')
    for nb in entry.get('notebooks', []):
        st = nb['status']
        ic = 'ok' if st == 'OK' else ('!!' if st == 'FAILED' else '..')
        t  = f'  ({fmt_time(nb["elapsed_s"])})' if 'elapsed_s' in nb else ''
        print(f'         [{ic}]  {nb["notebook"]}{t}')

print(f'\n  Total wall time : {fmt_time(total_s)}')

failed = [e['stage'] for e in log
          if e.get('status') == 'ran' and not e.get('success', True)]
if failed:
    print(f'  FAILED stages   : {failed}')
else:
    ran     = [e['stage'] for e in log if e.get('status') == 'ran']
    skipped = [e['stage'] for e in log if e.get('status') == 'skipped']
    if ran:
        print(f'  Ran             : {ran}')
    if skipped:
        print(f'  Skipped (fresh) : {skipped}')

PIPELINE SUMMARY
-------------------------------------------------------
  [RAN]  0 - Transformation  OK
         [ok]  01_energy_prices.ipynb  (2.2 min)
         [ok]  02_weather.ipynb  (15s)
  [RAN]  1 - Modelling  OK
         [ok]  01_solar_irradiation.ipynb  (4.0 min)
         [ok]  02_temperature.ipynb  (2.6 min)
         [ok]  03_noct_uplift.ipynb  (17s)
         [ok]  04_solar_share_historical.ipynb  (15s)
         [ok]  05_capture_rate.ipynb  (1.7 min)
         [ok]  06_copula.ipynb  (2.2 min)
  [RAN]  2 - Scenario Paths  OK
         [ok]  01_temperature_paths.ipynb  (17s)
         [ok]  02_solar_share_paths.ipynb  (1.2 min)
  [RAN]  3 - Simulation  OK
         [ok]  01_innovations.ipynb  (5.2 min)
         [ok]  02_variables_reconstruction.ipynb  (5.5 min)
  [RAN]  4 - Results  OK
         [ok]  03_revenue_index_historical.ipynb  (24s)
         [ok]  04_revenue_index_scenarios.ipynb  (3.7 min)
         [ok]  05_hedge_contract.ipynb  (2.6 min)

  Total wall time : 32.3 min
  Ra

## Post-run verification

Checks the invariants that a green run does not by itself prove. Each one
corresponds to a defect this pipeline has actually produced.

In [7]:
# ── Invariant checks on the artifacts this run produced ──────────────────────
# Gated on the run itself: if the pipeline halted, the artifacts on disk are
# whatever an EARLIER run left behind, and checking them would report a healthy
# chain that this run did not produce.
if not pipeline_ok:
    raise RuntimeError(
        'Pipeline did not complete, so these checks would be validating stale '
        'artifacts from a previous run. Fix the failed stage and re-run.')

import pickle

import pandas as pd

checks, failures = [], []


def check(name, ok, detail):
    checks.append((name, bool(ok), detail))
    if not ok:
        failures.append(name)


# 1. Every Stage-1 model artifact the simulation reads must exist.
_noct = CODE / 'Models' / 'Temperature' / 'noct_uplift_params.pkl'
check('NOCT uplift artifact exists', _noct.exists(), str(_noct.name))
if _noct.exists():
    with open(_noct, 'rb') as _f:
        _p = pickle.load(_f)
    _u = _p['uplift_by_doy']
    _jja = sum(_u[d] for d in range(172, 265)) / 93
    _djf = sum(_u[d] for d in list(range(1, 60)) + list(range(335, 367))) / 91
    check('uplift profile has 366 entries', len(_u) == 366, f'{len(_u)} entries')
    check('uplift is seasonal and positive', _jja > _djf and min(_u.values()) > 0,
          f'JJA {_jja:.2f} C > DJF {_djf:.2f} C')

# 2. The three scenario files must agree on path count.
_ns = {}
for _k in ('current_trend', 'pniec', 'entso_e'):
    _f = DATA / 'Simulated' / f'mc_physical_variables_{_k}.parquet'
    if _f.exists():
        import pyarrow.parquet as _pq
        _ns[_k] = _pq.ParquetFile(_f).metadata.num_rows
check('scenario files agree on row count', len(set(_ns.values())) == 1,
      ', '.join(f'{k}={v:,}' for k, v in _ns.items()))

# 3. THE defect this rework existed to fix: the results artifact once held six
#    scenario cells at 1,000 paths and three at 5,000, with nothing raising.
_ri = DATA / 'Results' / 'ri_annual_results_multiscenario.parquet'
if _ri.exists():
    _d = pd.read_parquet(_ri, columns=['scenario_id', 'sim_id', 'year'])
    _cells = _d.groupby('scenario_id', observed=True)['sim_id'].nunique()
    _paths = int(_cells.iloc[0])
    _years = _d['year'].nunique()
    check('9 scenario cells present', len(_cells) == 9, f'{len(_cells)} cells')
    check('uniform path count across scenarios', _cells.nunique() == 1,
          f'{sorted(set(_cells))} paths')
    check('row count is 9 x paths x years',
          len(_d) == 9 * _paths * _years,
          f'{len(_d):,} rows = 9 x {_paths:,} x {_years}')
else:
    check('multi-scenario results exist', False, 'file missing')

# 4. The historical benchmark must cover the observed record.
_rh = DATA / 'Results' / 'ri_annual_results_hist.parquet'
if _rh.exists():
    _h = pd.read_parquet(_rh)
    check('historical benchmark spans 2005-2025',
          int(_h['year'].min()) == 2005 and int(_h['year'].max()) == 2025,
          f"{int(_h['year'].min())}-{int(_h['year'].max())}, {len(_h)} rows")
else:
    check('historical benchmark exists', False, 'file missing')

# ── report ───────────────────────────────────────────────────────────────────
print('POST-RUN VERIFICATION')
print('-' * 68)
for _name, _ok, _detail in checks:
    print(f"  [{'ok' if _ok else '!!'}]  {_name:<42}  {_detail}")
print('-' * 68)
if failures:
    raise AssertionError(f'{len(failures)} invariant(s) failed: {failures}')
print(f'  All {len(checks)} invariants hold.')

POST-RUN VERIFICATION
--------------------------------------------------------------------
  [ok]  NOCT uplift artifact exists                 noct_uplift_params.pkl
  [ok]  uplift profile has 366 entries              366 entries
  [ok]  uplift is seasonal and positive             JJA 21.15 C > DJF 10.13 C
  [ok]  scenario files agree on row count           current_trend=45,655,000, pniec=45,655,000, entso_e=45,655,000
  [ok]  9 scenario cells present                    9 cells
  [ok]  uniform path count across scenarios         [5000] paths
  [ok]  row count is 9 x paths x years              1,125,000 rows = 9 x 5,000 x 25
  [ok]  historical benchmark spans 2005-2025        2005-2025, 21 rows
--------------------------------------------------------------------
  All 8 invariants hold.


## Conclusion & Handoff

- A clean run of this notebook regenerates every artifact the thesis quotes,
  from raw ingestion through the hedge premium, in dependency order.
- The post-run verification above is the only thing that certifies a run: the
  pipeline summary shows every stage as OK, not that the artifacts it wrote are
  internally consistent.

**This is the entry point.** No downstream notebook consumes this one's own
output; it exists to run every other notebook, in order, from a clean
checkout.